<a href="https://colab.research.google.com/github/marcory-hub/yolo11n-on-grove-vision-ai-v2/blob/main/YOLO11n_training_2026_02_18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLO11n training




Last accessed: 2026-02-19 to train dataset_vespa_2026-02v3.


1. Make sure images and labels from your dataset have this folder structure with these exact names. And add `data.yaml` to main folder.

```
🗂️ dataset
  🗂️ train
    🗂️ images
    🗂️ labels
  🗂️ valid
    🗂️ images
    🗂️ labels
  data.yaml
```

2. Zip the dataset folder to a file names `dataset.zip` On mac use `zip -r dataset.zip dataset -i '*.yaml'*.jpg' '*.txt' '*/'` to include yaml, jpg and txt only.

3. Copy the `dataset.zip` file to /`content/drive/MyDrive`, it is needed to make a callibration image set and your yolo model, fe `best.pt` to this folder. For the model you can use a custom name and adjust it in the options below.

4. Copy the `dataset.zip` file to the folder /content/drive/MyDrive/yolo.


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
#check GPU
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Fri Feb 20 22:59:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Adjust the zipped file name

In [ ]:
# Copy zipped dataset to colab and unzip the dataset
!cp '/content/drive/MyDrive/dataset.zip' '/content/dataset.zip'
!unzip '/content/dataset.zip' -d '/content/dataset/'

Streaminguitvoer ingekort tot de laatste 5000 regels.
  inflating: /content/dataset/train/labels/vespsp1_0188_jpg.rf.cac672633fc0e96d7ecb2a3456e3f3d9.txt  
  inflating: /content/dataset/train/labels/amel1_1673_jpg.rf.8e0a2c5480b0a7998935944775db2fb6.txt  
  inflating: /content/dataset/train/labels/vcra1_1741_jpg.rf.ef6683723269e9778b5cb78f79999826.txt  
  inflating: /content/dataset/train/labels/vespsp1_1139_jpg.rf.5e8ccd8a21d00de03d0d5a702c2c8cdc.txt  
  inflating: /content/dataset/train/labels/vvel2_3527_jpg.rf.0196adc3eb0d4559e8193a86f5aec9e6.txt  
  inflating: /content/dataset/train/labels/amel2_6953_jpg.rf.334c5287e99356384ccaabee625ee8a1.txt  
  inflating: /content/dataset/train/labels/vcra2_3562_jpg.rf.727cf2739dd2182da8fc03330083d91c.txt  
  inflating: /content/dataset/train/labels/vespsp2_3381_jpg.rf.63a64e7d1dc901511189f8ccfda412a7.txt  
  inflating: /content/dataset/train/labels/vespsp2_1073_jpg.rf.bd00d885d3a8cd1da2f0d277834560fd.txt  
  inflating: /content/dataset/train/la

In [ ]:
# Install the required packages for Ultralytics YOLO 8.3.166
# !pip install -U ultralytics==8.3.166

In [ ]:
# Install the required packages for Ultralytics YOLO and Weights & Biases
!pip install -U ultralytics==8.4.14 wandb



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 82.1 MB/s eta 0:00:00


In [ ]:
# check ultralytics version is 8.3.166/ 8.4.14
import ultralytics
print(ultralytics.__version__)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
8.4.14


In [ ]:
from ultralytics import YOLO

# Initialize YOLO
yolo = YOLO()

In [ ]:
!pip install -q wandb


# Train, zip and download the model

Action:
1. Adjust epochs (select 10 for testing purposes, > 200 for training)
2. Adust batch_size: common batch sizes that work well with GPU architectures are powers of two, such as 16, 32, 64, 128, ... 512. -1 for autobatch.
3. Set image size (default 192, max 224).
4. remove if these aumentations were done in the dataset:             
- fliplr=0.0,
- degrees=10.0,
- hsv_h=0.0,
- hsv_s=0.0,
- hsv_v=0.0,
- epochs=epochs,
- lr0=0.005,
- warmup_epochs=10,
- patience=30,


-----

In [ ]:
# Run this before model.train()
# !ln -s /content/drive/MyDrive/YOLO_trainings /content/vespa_2026-02

In [ ]:
import os
import shutil
from google.colab import userdata, drive
from ultralytics import YOLO, settings

# --- 1. Mount Drive ---
drive.mount('/content/drive')
drive_save_path = "/content/drive/MyDrive/YOLO_trainings"

# --- 2. Enable W&B in Ultralytics ---
os.environ["WANDB_API_KEY"] = userdata.get('wandb-key')
settings.update({"wandb": True})


# --- 3. Config ---
project_name = "vespa_2026-02"
name = "yolo11n_v1_40px_e300_b395_imgsz224"

# --- 4. Train ---
model = YOLO("yolo11n.pt")

model.train(
    data="/content/dataset/data.yaml",
    epochs=300,
    batch=395,
    imgsz=224,
    project=project_name,
    name=name,
    patience=30
)


# --- 6. Backup to Drive ---
final_local_path = f"{project_name}/{name}"
final_drive_path = f"{drive_save_path}/{project_name}/{name}"

os.makedirs(os.path.dirname(final_drive_path), exist_ok=True)
if os.path.exists(final_local_path):
    shutil.copytree(final_local_path, final_drive_path, dirs_exist_ok=True)
    print(f"Results backed up to: {final_drive_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=395, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_sc

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mvdijk (mvdijk-vespcv) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Overriding model.yaml nc=80 with nc=4

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

lr/pg0,▃████▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁
lr/pg1,▆███▇▇▇▇▇▆▆▆▆▅▅▅▅▄▄▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁
lr/pg2,████▇▇▇▇▇▆▆▆▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁
lr/pg3,▃████▇▇▇▇▇▆▆▆▆▆▆▆▆▆▅▅▅▅▄▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁
lr/pg4,█████▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁
lr/pg5,████▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▁▁▁▁▁
lr/pg6,▃████▇▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▄▃▃▂▂▂▂▁▁▁▁▁
lr/pg7,█████▇▇▇▆▆▆▆▆▆▆▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▁
metrics/mAP50(B),▂▄▁▇▇▇▇▇▇███████████████████████████████
metrics/mAP50-95(B),▁▁▂▂▃▃▄▅▅▅▆▆▆▆▆▆▇▆▇▇▇▇▇█████████████████
+11,...


In [ ]:
import shutil
from google.colab import files

# --- Paths ---
source_folder = "/content/runs/detect/vespa_2026-02"
zip_file = "/content/vespa_2026-02.zip"

# --- Create ZIP ---
shutil.make_archive(zip_file.replace('.zip',''), 'zip', source_folder)
print(f"✅ Folder zipped to {zip_file}")

# --- Download to local machine ---
files.download(zip_file)


✅ Folder zipped to /content/vespa_2026-02.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Kill runtime
# import os
# os.kill(os.getpid(), 9)